<img src="https://raw.githubusercontent.com/OmarLi-360/excel-corporativo-12-claves/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 11 · Gráficos y narrativa de resultados
**Excel Corporativo · 12 Claves** — Del dato al informe que se explica solo · ⏱ 30 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/11_graficos_y_narrativa.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-11)

<sub>Si al abrir un enlace Google muestra «Redirect Notice», haz clic en el enlace azul de esa página: es un aviso de seguridad de Google, no un error.</sub>

**Al terminar podrás:**
1. Comparar el gráfico por defecto con uno diseñado para responder una pregunta.
2. Elegir el gráfico y el título según el mensaje del informe.
3. Armar un informe de una página: qué pasó, por qué y qué sigue.

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden. Si no lo haces, no pasa nada grave: el original está protegido y nadie puede dañarlo desde aquí, pero lo que cambies se borra al cerrar.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/excel-corporativo-12-claves"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

In [ ]:
# @title Preparar real y meta 2025 por sede
real = validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
real24 = validas[validas.anio == 2024].groupby("ciudad")["subtotal"].sum()
meta = (metas[metas.anio == 2025].merge(sedes[["id_sede", "ciudad"]], on="id_sede").groupby("ciudad")["meta_subtotal"].sum())
tabla = pd.DataFrame({"real": real, "meta": meta, "real_2024": real24}).reindex(sedes.ciudad)
tabla["cumplimiento"] = tabla.real / tabla.meta
tabla["crecimiento"] = tabla.real / tabla.real_2024 - 1
display(tabla.style.format({"real": miles, "meta": miles, "real_2024": miles, "cumplimiento": "{:.1%}", "crecimiento": "{:+.1%}"}))

## 1 · Lo que sale por defecto
La pregunta del comité es: **«¿qué sedes cumplieron la meta de 2025?»**. Este es el gráfico que muchos insertan al seleccionar la tabla: columnas de real y meta, en el orden de la tabla, con un título que nombra el tema.

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Responde el gráfico por defecto la pregunta? |
> | **Qué se grafica** | Columnas agrupadas de real y meta por sede, sin ordenar · Título genérico |
> | **Por qué esta forma** | Se muestra a propósito tal como sale por defecto, para compararlo con la versión siguiente. |
> | **Qué se descartó** | — |
> | **Cómo se lee** | Intenta decir en cinco segundos qué sedes cumplieron: hay que comparar pares de columnas una por una. |
> | **En Excel 365** | Es lo que da Insertar › Gráfico recomendado sobre la tabla sin ninguna decisión. |

In [ ]:
# @title Gráfico 1 · Versión por defecto
fig = go.Figure()
fig.add_bar(name="Real", x=tabla.index, y=tabla.real / 1e6)
fig.add_bar(name="Meta", x=tabla.index, y=tabla.meta / 1e6)
fig.update_layout(barmode="group", title="Ventas por sede 2025", yaxis_title=MM, height=360)
fig.show()

## 2 · El gráfico según el mensaje
Primero se decide **qué se quiere decir**; el gráfico y el título vienen después. Elige el mensaje del informe:

* **Cumplimiento de la meta**: barras de % ordenadas, línea en 100 % y color solo para las que no cumplen.
* **Crecimiento frente a 2024**: barras de crecimiento ordenadas y una línea con el crecimiento total.
* **Mezcla de canales**: barras apiladas al 100 % con el canal que importa en color.

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué gráfico y qué título responden el mensaje elegido? |
> | **Qué se grafica** | Depende del mensaje: barras de cumplimiento, barras de crecimiento o barras apiladas al 100 % |
> | **Por qué esta forma** | Cada mensaje es una pregunta distinta (cumplimiento, cambio o composición) y cada una tiene su forma. |
> | **Qué se descartó** | Usar el mismo gráfico para los tres mensajes: alguno quedaría sin respuesta. |
> | **Cómo se lee** | El título dice el hallazgo con su cifra; el color solo marca lo que el título menciona. |
> | **En Excel 365** | Barras ordenadas, una serie de referencia como línea, relleno gris y un color de énfasis, título enlazado a una celda. |

In [ ]:
# @title Gráfico 2 · Mensaje, gráfico y título
mensaje = "Cumplimiento de la meta"  # @param ["Cumplimiento de la meta", "Crecimiento frente a 2024", "Mezcla de canales"]
if mensaje == "Cumplimiento de la meta":
    t = tabla.sort_values("cumplimiento")
    colores = [MAGENTA if c < 1 else GRIS for c in t.cumplimiento]
    fig = go.Figure(go.Bar(x=t.cumplimiento, y=t.index, orientation="h", marker_color=colores,
                           text=[f"{c:.1%}".replace(".", ",").replace("%", " %") for c in t.cumplimiento], textposition="outside", cliponaxis=False))
    fig.add_vline(x=1, line_dash="dash", line_color="#11153A", annotation_text="Meta 100 %", annotation_position="top")
    cumplen = t[t.cumplimiento >= 1].index.tolist()[::-1]
    peor = t.index[0]
    titulo = f"{' y '.join(cumplen)} superaron su meta; {peor} quedó {(1 - t.cumplimiento.iloc[0]) * 100:.0f} puntos abajo"
    fig.update_layout(xaxis_tickformat=".0%", xaxis_range=[0, t.cumplimiento.max() * 1.15])
elif mensaje == "Crecimiento frente a 2024":
    t = tabla.sort_values("crecimiento")
    total = tabla.real.sum() / tabla.real_2024.sum() - 1
    colores = [AZUL if c > total else GRIS for c in t.crecimiento]
    fig = go.Figure(go.Bar(x=t.crecimiento, y=t.index, orientation="h", marker_color=colores,
                           text=[porc(c, signo=True) for c in t.crecimiento], textposition="outside", cliponaxis=False))
    fig.add_vline(x=total, line_dash="dash", line_color="#11153A", annotation_text=f"Total {porc(total, signo=True)}", annotation_position="top")
    titulo = f"{t.index[-1]} creció {porc(t.crecimiento.iloc[-1])}, más del doble que el total de la empresa ({porc(total)})" if t.crecimiento.iloc[-1] > 2 * total \
             else f"{t.index[-1]} lideró el crecimiento con {porc(t.crecimiento.iloc[-1])}"
    fig.update_layout(xaxis_tickformat=".0%", xaxis_range=[0, t.crecimiento.max() * 1.2])
else:
    mezcla = (validas[validas.anio == 2025].pivot_table(index="ciudad", columns="canal", values="subtotal", aggfunc="sum")
              .reindex(columns=["E-commerce", "Televentas", "Presencial"]))
    mezcla = mezcla.div(mezcla.sum(axis=1), axis=0).sort_values("E-commerce")
    fig = go.Figure()
    for canal, color in [("E-commerce", AZUL), ("Televentas", "#C9D0E2"), ("Presencial", "#E4E8F2")]:
        fig.add_bar(name=canal, y=mezcla.index, x=mezcla[canal], orientation="h", marker_color=color,
                    text=[porc(v) if canal == "E-commerce" else "" for v in mezcla[canal]], textposition="inside")
    titulo = f"E-commerce pesa {porc(mezcla['E-commerce'].iloc[-1])} de la venta en {mezcla.index[-1]} y {porc(mezcla['E-commerce'].iloc[0])} en {mezcla.index[0]}"
    fig.update_layout(barmode="stack", xaxis_tickformat=".0%", legend=dict(orientation="h", y=-0.15))
fig.update_layout(title=titulo, height=380, showlegend=(mensaje == "Mezcla de canales"))
fig.show()
display(Markdown(f"**Título del hallazgo:** {titulo}"))

## 3 · Informe de una página

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cómo se ve el informe completo para el comité? |
> | **Qué se grafica** | Indicador de cumplimiento total con su diferencia frente a la meta, y líneas de real y meta mes a mes |
> | **Por qué esta forma** | El indicador da la respuesta en un número; la línea muestra cuándo se ganó o se perdió. |
> | **Qué se descartó** | Un tablero con diez gráficos: el comité necesita un mensaje, no todos los datos. |
> | **Cómo se lee** | Lee primero el número grande, luego busca los meses donde la línea real queda debajo de la meta. |
> | **En Excel 365** | Hoja Panel: una tarjeta con =SUMA(…)/SUMA(…), un gráfico de líneas y un cuadro de texto enlazado a la narrativa. |

In [ ]:
# @title Gráfico 3 · Informe de una página
from plotly.subplots import make_subplots
mens = validas[validas.anio == 2025].groupby("mes")["subtotal"].sum() / 1e6
mmeta = metas[metas.anio == 2025].groupby("mes")["meta_subtotal"].sum() / 1e6
fechas = pd.to_datetime([f"2025-{m:02d}-01" for m in mens.index])
fig = make_subplots(rows=1, cols=2, column_widths=[0.3, 0.7], specs=[[{"type": "indicator"}, {"type": "xy"}]])
fig.add_trace(go.Indicator(mode="number+delta", value=tabla.real.sum() / tabla.meta.sum() * 100,
                           number=dict(suffix=" %", valueformat=".1f"), delta=dict(reference=100, valueformat=".1f", suffix=" pts"),
                           title=dict(text="Cumplimiento 2025")), row=1, col=1)
fig.add_scatter(x=fechas, y=mens.values, name="Real", line=dict(color=AZUL, width=3), row=1, col=2)
fig.add_scatter(x=fechas, y=mmeta.values, name="Meta", line=dict(color="#11153A", width=2, dash="dash"), row=1, col=2)
debajo = int((mens.values < mmeta.values).sum())
eje_meses(fig, fechas)
fig.update_layout(title=f"La empresa cerró 2025 en {porc(tabla.real.sum() / tabla.meta.sum())} de la meta: {debajo} de 12 meses quedaron debajo",
                  height=380, yaxis_title=MM, legend=dict(orientation="h", y=-0.15))
fig.show()
peor = tabla.cumplimiento.idxmin(); mejor = tabla.cumplimiento.idxmax()
display(Markdown(f"""
**Qué pasó.** La venta neta de 2025 fue {millones(tabla.real.sum())}, el {porc(tabla.real.sum() / tabla.meta.sum())} de la meta.

**Por qué.** {mejor} superó su meta ({porc(tabla.cumplimiento[mejor])}) y creció {porc(tabla.crecimiento[mejor])} frente a 2024; {peor} quedó en {porc(tabla.cumplimiento[peor])}.

**Qué sigue.** Revisar con {peor} la cartera de clientes y el plan del primer trimestre, y replicar lo que funcionó en {mejor}.
"""))

## 4 · Retos de interpretación

In [ ]:
# @title Reto 1 · ¿Cuál de estos es un título de hallazgo?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Ventas por sede 2025", "Cumplimiento de metas", "Barranquilla quedó 12 puntos debajo de su meta", "Gráfico de barras"]
revisar(respuesta, "Barranquilla quedó 12 puntos debajo de su meta", """Dice qué pasó y con qué cifra; los otros solo nombran el tema.""")

In [ ]:
# @title Reto 2 · ¿Qué sede quedó más lejos de su meta en 2025?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Bogotá", "Medellín", "Cali", "Barranquilla", "Bucaramanga", "Pereira"]
revisar(respuesta, tabla.cumplimiento.idxmin(), """Es la barra más corta del gráfico de cumplimiento.""")

In [ ]:
# @title Reto 3 · Quieres mostrar cómo evolucionó la venta mes a mes. ¿Qué gráfico usas?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Torta", "Líneas", "Barras apiladas al 100 %", "Dispersión"]
revisar(respuesta, "Líneas", """La evolución en el tiempo se lee mejor como una línea continua.""")

## Apóyate en IA
Para pasar de la tabla a la narrativa:

```text
Esta es la tabla de real, meta y cumplimiento 2025 por sede: [PEGA LA TABLA].
Escribe tres títulos de hallazgo de máximo 12 palabras con cifra, y la narrativa en tres frases: qué pasó, por qué y qué sigue.
No inventes datos que no estén en la tabla.
```

> **Verifica siempre.** Compara la respuesta de la IA con un dato que ya conozcas. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* Primero el mensaje, después el gráfico.
* Ordenar, un color de énfasis, una referencia y etiquetas directas hacen que se entienda en diez segundos.
* El título dice el hallazgo; la narrativa dice qué pasó, por qué y qué sigue.

**Siguiente:** [Clave 12 · Auditoría y control](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-12) · [Cuaderno 12](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/12_auditoria_y_control.ipynb)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>